In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, GRU, SimpleRNN
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ==========================================================
# 1️⃣ PATH SETTINGS
# ==========================================================

data_path = r"C:\Users\abhis\OneDrive\Desktop\Abhishek2\125\Final_Cleaned_Modeling_Dataset.csv"
result_path = r"C:\Users\abhis\OneDrive\Desktop\Abhishek2\125"

models_list = ["LSTM", "BiLSTM", "GRU", "RNN", "CNN"]

for model_name in models_list:
    for folder in ["eval", "loss", "pred", "model"]:
        os.makedirs(os.path.join(result_path, model_name, folder), exist_ok=True)

# ==========================================================
# 2️⃣ LOAD DATA
# ==========================================================

df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y %H:%M')
df = df.sort_values('Date')
df.set_index('Date', inplace=True)

print("Dataset Loaded Successfully")
print(df.head())

# ==========================================================
# 3️⃣ TRAIN-TEST SPLIT (NO DATA LEAKAGE)
# ==========================================================

train_size = int(0.8 * len(df))
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df)
test_scaled = scaler.transform(test_df)

# ==========================================================
# 4️⃣ CREATE SEQUENCES
# ==========================================================

TIME_STEPS = 10

def create_sequences(data, time_steps=10):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, TIME_STEPS)
X_test, y_test = create_sequences(test_scaled, TIME_STEPS)

# Time-aware validation split
val_split = int(0.8 * len(X_train))
X_tr, X_val = X_train[:val_split], X_train[val_split:]
y_tr, y_val = y_train[:val_split], y_train[val_split:]

# ==========================================================
# 5️⃣ MODEL BUILDER
# ==========================================================

def build_model(model_type):

    model = Sequential()

    if model_type == "LSTM":
        model.add(LSTM(64, return_sequences=True,
                       input_shape=(TIME_STEPS, X_train.shape[2])))
        model.add(Dropout(0.3))
        model.add(LSTM(32))
        model.add(Dropout(0.3))

    elif model_type == "BiLSTM":
        model.add(Bidirectional(LSTM(64, return_sequences=True),
                                input_shape=(TIME_STEPS, X_train.shape[2])))
        model.add(Dropout(0.3))
        model.add(Bidirectional(LSTM(32)))
        model.add(Dropout(0.3))

    elif model_type == "GRU":
        model.add(GRU(64, return_sequences=True,
                      input_shape=(TIME_STEPS, X_train.shape[2])))
        model.add(Dropout(0.3))
        model.add(GRU(32))
        model.add(Dropout(0.3))

    elif model_type == "RNN":
        model.add(SimpleRNN(64, return_sequences=True,
                            input_shape=(TIME_STEPS, X_train.shape[2])))
        model.add(Dropout(0.3))
        model.add(SimpleRNN(32))
        model.add(Dropout(0.3))

    elif model_type == "CNN":
        model.add(Conv1D(filters=64, kernel_size=2,
                         activation='relu',
                         input_shape=(TIME_STEPS, X_train.shape[2])))
        model.add(MaxPooling1D(pool_size=2))
        model.add(Flatten())
        model.add(Dense(32, activation='relu'))

    model.add(Dense(y_train.shape[1]))
    model.compile(optimizer='adam', loss='mse')

    return model

# ==========================================================
# 6️⃣ EVALUATION FUNCTION (WITH DATE + ACTUAL + PREDICTED)
# ==========================================================

def evaluate_model(model_name, model):

    y_pred = model.predict(X_test)

    # Inverse transform
    y_test_inv = scaler.inverse_transform(y_test)
    y_pred_inv = scaler.inverse_transform(y_pred)

    # Correct corresponding dates
    test_dates = test_df.index[TIME_STEPS:]

    pred_df = pd.DataFrame({"Date": test_dates})

    for i, col in enumerate(df.columns):
        pred_df[f"Actual_{col}"] = y_test_inv[:, i]
        pred_df[f"Predicted_{col}"] = y_pred_inv[:, i]

    pred_df.to_csv(
        os.path.join(result_path, model_name, "pred", "predictions_with_date.csv"),
        index=False
    )

    # ================= Metrics =================

    metrics_dict = {}

    for i, col in enumerate(df.columns):
        mse = mean_squared_error(y_test_inv[:, i], y_pred_inv[:, i])
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test_inv[:, i], y_pred_inv[:, i])
        mape = np.mean(
            np.abs((y_test_inv[:, i] - y_pred_inv[:, i]) /
                   np.maximum(np.abs(y_test_inv[:, i]), 1e-8))
        ) * 100
        r2 = r2_score(y_test_inv[:, i], y_pred_inv[:, i])

        metrics_dict[col] = {
            "MSE": mse,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "R2": r2
        }

    with open(os.path.join(result_path, model_name, "eval", "metrics.json"), "w") as f:
        json.dump(metrics_dict, f, indent=4)

    pd.DataFrame(metrics_dict).T.to_csv(
        os.path.join(result_path, model_name, "eval", "metrics.csv")
    )

# ==========================================================
# 7️⃣ TRAIN ALL MODELS
# ==========================================================

for model_name in models_list:

    print(f"\nTraining {model_name}...")

    model = build_model(model_name)

    early_stop = EarlyStopping(patience=10, restore_best_weights=True)
    lr_reduce = ReduceLROnPlateau(patience=5, factor=0.5)

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        callbacks=[early_stop, lr_reduce],
        verbose=1
    )

    # Save loss plot
    plt.figure()
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.legend()
    plt.title(f"{model_name} Loss Curve")
    plt.savefig(os.path.join(result_path, model_name, "loss", "loss_plot.png"))
    plt.close()

    # Save model
    model.save(os.path.join(result_path, model_name, "model", f"{model_name}.h5"))

    # Evaluate
    evaluate_model(model_name, model)

print("\n✅ All Models Trained Successfully")
print("Results saved in:", result_path)

Dataset Loaded Successfully
            PM2.5    PM10     NO2
Date                             
2024-03-01  56.43  167.86  101.20
2024-03-02  15.97   50.54   15.99
2024-03-03  27.39   81.64   25.36
2024-03-04  21.21   65.71   32.76
2024-03-05  30.55   83.86   38.41

Training LSTM...
Epoch 1/100


C:\Users\abhis\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - loss: 0.0343 - val_loss: 0.0090 - learning_rate: 0.0010
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0186 - val_loss: 0.0050 - learning_rate: 0.0010
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0186 - val_loss: 0.0065 - learning_rate: 0.0010
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0183 - val_loss: 0.0046 - learning_rate: 0.0010
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0172 - val_loss: 0.0055 - learning_rate: 0.0010
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0169 - val_loss: 0.0052 - learning_rate: 0.0010
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0172 - val_loss: 0.0047 - learning_rate: 0.0010
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0163 - val_loss: 0.0057 - learning_rate: 0.0010
Epoch 9/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0160 - val_loss: 0.0044 - learning_rate: 0.0010
Epoch 10/100
15/15 ━━━

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step

Training BiLSTM...
Epoch 1/100


C:\Users\abhis\anaconda3\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.1935 - val_loss: 0.0143 - learning_rate: 0.0010
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1019 - val_loss: 0.0087 - learning_rate: 0.0010
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0734 - val_loss: 0.0064 - learning_rate: 0.0010
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0572 - val_loss: 0.0055 - learning_rate: 0.0010
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0486 - val_loss: 0.0058 - learning_rate: 0.0010
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0422 - val_loss: 0.0052 - learning_rate: 0.0010
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0391 - val_loss: 0.0047 - learning_rate: 0.0010
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0339 - val_loss: 0.0047 - learning_rate: 0.0010
Epoch 9/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0313 - val_loss: 0.0045 - learning_rate: 0.0010
Epoch 10/100
15/15 ━━━━━━━━━━━

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step

Training CNN...
Epoch 1/100


C:\Users\abhis\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0366 - val_loss: 0.0060 - learning_rate: 0.0010
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0191 - val_loss: 0.0077 - learning_rate: 0.0010
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0171 - val_loss: 0.0068 - learning_rate: 0.0010
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0161 - val_loss: 0.0060 - learning_rate: 0.0010
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0152 - val_loss: 0.0054 - learning_rate: 0.0010
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0149 - val_loss: 0.0045 - learning_rate: 0.0010
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0144 - val_loss: 0.0052 - learning_rate: 0.0010
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0143 - val_loss: 0.0045 - learning_rate: 0.0010
Epoch 9/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0137 - val_loss: 0.0045 - learning_rate: 0.0010
Epoch 10/100
15/15 ━━━━━━━━━━━

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step

✅ All Models Trained Successfully
Results saved in: C:\Users\abhis\OneDrive\Desktop\Abhishek2\125
